In [1]:
%load_ext autoreload
%autoreload 2
import scanpy as sc
import snapatac2 as snap
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Spatial Expression of Genes Across Samples

In [2]:
batches = ["gw6", "gw7-2", "gw9", "gw10", "gw12", "gw16", "gw17-1"]
sample_key = "batch"
spot_size = 10

In [19]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.colors import LinearSegmentedColormap

# Gradient colormap
# exp_colors = ['white', "#EDEDED", "#FFF7F3","#F7FCF0","#E0F3DB","#CCEBC5","#7BCCC4","#4EB3D3","#2B8CBE","#0868AC","#084081"]
# exp_cmap = LinearSegmentedColormap.from_list("my_cmap", exp_colors, N=256)
exp_colors = ["#EDEDED", "#FFF7F3","#F7FCF0","#E0F3DB","#CCEBC5","#7BCCC4","#4EB3D3","#2B8CBE","#0868AC","#084081"]
exp_cmap = LinearSegmentedColormap.from_list("my_cmap", exp_colors, N=256)
exp_cmap

RNA_colors = ["#352A86", "#343296", "#343AA7", "#2646BB", "#1156D0", "#0366DE", "#0A76D8", "#1286D2", "#1C99C0", "#27ADAC", "#46B897", "#78BB7E", "#A9BD67", "#CCBC57", "#EFBA47", "#F7C438", "#F6D12B", "#F6DF1F", "#F7EC16", "#F8FA0D"]
RNA_cmap = LinearSegmentedColormap.from_list("my_cmap", RNA_colors, N=256)

In [4]:
file_path = "/cluster2/huanglab/jiamao/Project/Tools/nichecompass-reproducibility/artifacts/stereo_seq_spinal_cord_gwall/results/single_sample/27052025_044245/stereo_seq_spinal_cord_gwall_analysis.h5ad"
adata = sc.read_h5ad(file_path)
adata

AnnData object with n_obs × n_vars = 82934 × 8000
    obs: 'nCount_RNA', 'nFeature_RNA', 'nCount_Spatial', 'nFeature_Spatial', 'cell', 'x', 'y', 'batch', 'latent_leiden_0.3', 'Add-on_31_GP', 'Add-on_72_GP', 'PTPN11_ligand_receptor_GP', 'SERPING1_ligand_receptor_target_gene_GP', 'INHBA_combined_GP', 'FGF18_combined_GP', 'ST6GAL1_ligand_receptor_target_gene_GP', 'BST2_combined_GP', 'VSIG10_ligand_receptor_target_gene_GP', 'Heme_metabolite_enzyme_sensor_GP', 'CCL2_combined_GP', 'TGFB2_combined_GP', 'LMAN1_ligand_receptor_GP', 'LEPR_ligand_receptor_GP', 'ITIH2_ligand_receptor_target_gene_GP', 'ANGPTL1_combined_GP', 'IL19_combined_GP', 'IFITM1_ligand_receptor_target_gene_GP', 'COL15A1_combined_GP', 'SDC4_ligand_receptor_target_gene_GP', 'PGF_combined_GP', 'LAMC3_ligand_receptor_target_gene_GP', 'ILDR2_ligand_receptor_target_gene_GP', 'COL1A2_ligand_receptor_target_gene_GP', 'COL3A1_ligand_receptor_target_gene_GP', 'SPON2_combined_GP', 'L-Tyrosine_metabolite_enzyme_sensor_GP', 'IGSF11_ligand

In [5]:
cod = adata.obs[['x', 'y']]
# cod.to_csv('spatial_coord.csv', index=True)

In [6]:
# Configure paths and batches
so_data_folder_path = '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/CalculateCells/Smooth/H5AD'
batches = ["gw6", "gw7-2", "gw9", "gw10", "gw12", "gw16", "gw17-1"]
sample_key = "batch"

# Reload raw data (only to retrieve raw counts)
raw_adata_list = []

for batch in batches:
    # Read raw H5AD
    temp_adata = sc.read_h5ad(f"{so_data_folder_path}/SeuratObj_{batch}.h5ad")
    temp_adata.obs[sample_key] = batch
    raw_adata_list.append(temp_adata)

# Merge raw data
adata_full = ad.concat(raw_adata_list, join="outer", fill_value=0)

# Align cells
# This step aligns automatically using adata.obs_names
adata_full = adata_full[adata.obs_names].copy()
adata_full.obs = adata.obs.copy()
adata_full.obsm = adata.obsm.copy()
adata_full.obsp = adata.obsp.copy()
adata_full.uns = adata.uns.copy()
adata_full.layers['counts'] = adata_full.X.copy()

# Normalize and log-transform (for clearer visualization)
sc.pp.normalize_total(adata_full, target_sum=1e4)
sc.pp.log1p(adata_full)


# adata_full.raw = adata_full

# Replacement completed
adata = adata_full

In [7]:
adata.obs

,nCount_RNA,nFeature_RNA,nCount_Spatial,nFeature_Spatial,cell,x,y,batch,latent_leiden_0.3,Add-on_31_GP,...,VSTM1_ligand_receptor_target_gene_GP_source_score,VSTM1_ligand_receptor_target_gene_GP_target_score,LGALS8_ligand_receptor_GP_source_score,LGALS8_ligand_receptor_GP_target_score,Iron_metabolite_enzyme_sensor_GP_source_score,Iron_metabolite_enzyme_sensor_GP_target_score,13-cis-Retinoic acid_metabolite_enzyme_sensor_GP_source_score,13-cis-Retinoic acid_metabolite_enzyme_sensor_GP_target_score,SEMA7A_combined_GP_source_score,SEMA7A_combined_GP_target_score
GW7:4470_6510,1069.0,613,1069.0,613,GW7:4470_6510,4470.0,6510.0,gw6,8,3.316699,...,0.0,0.000000,0.0,0.0,0.000268,0.00000,0.0,0.0,0.0,0.000033
GW7:4680_6420,999.0,612,999.0,612,GW7:4680_6420,4680.0,6420.0,gw6,8,3.206848,...,0.0,0.000328,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000026
GW7:5280_5610,972.0,609,972.0,609,GW7:5280_5610,5280.0,5610.0,gw6,8,2.840173,...,0.0,0.000353,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000
GW7:4890_5580,1120.0,764,1120.0,764,GW7:4890_5580,4890.0,5580.0,gw6,6,0.056684,...,0.0,0.000000,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000637
GW7:5190_6510,859.0,577,859.0,577,GW7:5190_6510,5190.0,6510.0,gw6,8,1.019332,...,0.0,0.000661,0.0,0.0,0.000000,0.00123,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GW17:14550_23100,2.0,1,2.0,1,GW17:14550_23100,14550.0,23100.0,gw17-1,1,-2.469773,...,0.0,0.000000,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000
GW17:14580_23100,1.0,1,1.0,1,GW17:14580_23100,14580.0,23100.0,gw17-1,1,-0.777751,...,0.0,0.000000,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000
GW17:13980_23010,2.0,1,2.0,1,GW17:13980_23010,13980.0,23010.0,gw17-1,1,-1.988651,...,0.0,0.000000,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000
GW17:15240_17130,2.0,1,2.0,1,GW17:15240_17130,15240.0,17130.0,gw17-1,1,-1.671806,...,0.0,0.000000,0.0,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000


In [8]:
adata.obs[sample_key]

GW7:4470_6510          gw6
GW7:4680_6420          gw6
GW7:5280_5610          gw6
GW7:4890_5580          gw6
GW7:5190_6510          gw6
                     ...  
GW17:14550_23100    gw17-1
GW17:14580_23100    gw17-1
GW17:13980_23010    gw17-1
GW17:15240_17130    gw17-1
GW17:13980_16710    gw17-1
Name: batch, Length: 82934, dtype: category
Categories (7, object): ['gw6', 'gw7-2', 'gw9', 'gw10', 'gw12', 'gw16', 'gw17-1']

In [9]:
adata.layers['counts']

<82934x47454 sparse matrix of type '<class 'numpy.float32'>'
	with 29321497 stored elements in Compressed Sparse Row format>

In [28]:
# Spatial Expression of Genes

# 1. Define mapping between sample and spot_size
# Note: dictionary keys must exactly match categories in adata.obs['batch']
size_dict = {
    "gw6": 23,
    "gw7-2": 14,
    "gw9": 9,
    "gw10": 4,
    "gw12": 2.6,
    "gw16": 2,
    "gw17-1": 1.1
}

# Gene list
gene_list = ['NEFM', 'NEFL', 'NEFH']

# Plot parameters
sc.settings.set_figure_params(dpi=300, facecolor='white')

# Get all samples
sample_key = 'batch'
all_samples = adata.obs[sample_key].cat.categories.tolist()
n_genes = len(gene_list)
n_samples = len(all_samples)

# Compute figure size
fig_width = 4 * n_samples 
fig_height = 4 * n_genes
fig, axes = plt.subplots(n_genes, n_samples, figsize=(fig_width, fig_height))

# Create PDF file
with PdfPages('./plots/HSC_gene_spatial_nefm.pdf') as pdf:
    for gene_idx, gene in enumerate(gene_list):
        for sample_idx, sample in enumerate(all_samples):
            # --- Axis object of current subplot ---
            if n_genes > 1 and n_samples > 1:
                ax = axes[gene_idx, sample_idx]
            elif n_genes > 1:
                ax = axes[gene_idx]
            elif n_samples > 1:
                ax = axes[sample_idx]
            else:
                ax = axes
            
            # --- Get sample-specific spot_size ---
            # If sample is not in the dictionary, default to 10
            current_spot_size = size_dict.get(sample, 10)
            
            # --- Subset data for the current sample ---
            sample_mask = adata.obs[sample_key] == sample
            sample_data = adata[sample_mask]
            
            # --- Core logic: check whether the gene exists ---
            gene_found = (gene in sample_data.var_names) or (gene in sample_data.obs.columns)
            
            if gene_found:
                try:
                    sc.pl.embedding(
                        adata=sample_data,
                        basis='spatial',
                        color=gene,  
                        cmap=RNA_cmap, 
                        size=current_spot_size, # Use dynamically retrieved size
                        marker='s',             # Draw square markers
                        edgecolor='none',
                        title='', 
                        vmax=5,
                        frameon=False,
                        ax=ax,
                        show=False
                    )
                    ax.set_aspect('equal') # Keep square markers undistorted
                except Exception as e:
                    print(f"Skipping {gene} in {sample}: {e}")
                    gene_found = False 
            
            # --- Handle empty cases ---
            if not gene_found:
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)

            # --- Set labels consistently ---
            if sample_idx == 0:
                ax.text(-0.1, 0.5, gene, 
                        transform=ax.transAxes, 
                        fontsize=12, fontweight='bold', 
                        va='center', ha='right', rotation='vertical')
            else:
                ax.set_ylabel('')
            
            if gene_idx == 0:
                ax.set_title(f"{sample}", fontsize=12, fontweight='bold')
            else:
                ax.set_title('')

            ax.set_xticks([])
            ax.set_yticks([])

    # Adjust layout
    plt.suptitle(f"Spatial Expression of Genes", fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Save to PDF
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()
    plt.close()